In [ ]:
# 导入必要的库
import tensorflow as tf
import h5py
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import pandas as pd
from scipy.optimize import minimize

In [ ]:
# 探索数据集A的结构
def explore_mat_file(file_path):
    """探索mat文件的结构并显示关键信息"""
    print(f"开始分析文件: {file_path}")
    f = h5py.File(file_path, 'r')
    
    # 查看所有key
    print("\n文件中的所有键:")
    for key in f.keys():
        print(f"- {key}")
        dataset = f[key]
        print(f"  形状: {dataset.shape}, 数据类型: {dataset.dtype}")
        
        # 如果数据是合理大小，显示统计信息
        if len(dataset.shape) > 0 and np.prod(dataset.shape) < 10000000:
            data_array = np.array(dataset)
            if data_array.ndim > 1:
                data_array = data_array.transpose()  # 转置以与原代码一致
            print(f"  统计信息: min={np.min(data_array)}, max={np.max(data_array)}, mean={np.mean(data_array)}, std={np.std(data_array)}")
    f.close()

# 请确保更改下面的文件路径为您的实际路径
file_path = 'TRAIN38.mat'  # 请修改为真实路径
if os.path.exists(file_path):
    explore_mat_file(file_path)
else:
    print(f"文件不存在: {file_path}")
    print("请设置正确的文件路径")

In [ ]:
# 探索数据集B的结构
def explore_directory_structure(base_dir):
    """探索目录结构并显示关键信息"""
    print(f"开始分析目录: {base_dir}")
    
    # 检查目录是否存在
    if not os.path.exists(base_dir):
        print(f"目录不存在: {base_dir}")
        return
    
    # 查找所有子目录
    subdirs = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    print(f"\n发现子目录: {subdirs}")
    
    # 对于每个子目录，检查其中的npy文件
    for subdir in subdirs:
        subdir_path = os.path.join(base_dir, subdir)
        npy_files = glob.glob(os.path.join(subdir_path, "*.npy"))
        
        print(f"\n子目录 {subdir} 中发现 {len(npy_files)} 个npy文件")
        
        # 随机抽取一些文件来查看
        if npy_files:
            sample_size = min(3, len(npy_files))
            samples = np.random.choice(npy_files, sample_size, replace=False)
            
            for sample_file in samples:
                print(f"\n分析文件: {os.path.basename(sample_file)}")
                data = np.load(sample_file)
                print(f"  形状: {data.shape}, 数据类型: {data.dtype}")
                print(f"  统计信息: min={np.min(data)}, max={np.max(data)}, mean={np.mean(data)}, std={np.std(data)}")
                
    # 查找label_index.txt文件
    for subdir in subdirs:
        index_file = os.path.join(base_dir, subdir, "label_index.txt")
        if os.path.exists(index_file):
            print(f"\n找到索引文件: {index_file}")
            # 读取前几行
            with open(index_file, 'r') as f:
                lines = f.readlines()[:10]  # 读取前10行
                print("文件内容预览:")
                for line in lines:
                    print(f"  {line.strip()}")

# 请确保更改下面的目录路径为您的实际路径
restructured_base_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured'  # 请修改为真实路径
if os.path.exists(restructured_base_dir):
    explore_directory_structure(restructured_base_dir)
else:
    print(f"目录不存在: {restructured_base_dir}")
    print("请设置正确的目录路径")

In [ ]:
# 加载并处理数据集A
def load_dataset_A(file_path):
    """加载原始数据集A并进行基本处理"""
    print(f"加载数据集A: {file_path}")
    
    try:
        f = h5py.File(file_path, 'r')
        arrays = {}
        for k, v in f.items():
            arrays[k] = np.array(v)
        f.close()
        
        # 按照原代码的处理逻辑
        train_data = arrays['data'].transpose()
        train_region = arrays['region'].transpose()
        prob_idx = arrays['prob_idx'].transpose()
        
        print(f"数据集A加载完成:")
        print(f"  train_data 形状: {train_data.shape}")
        print(f"  train_region 形状: {train_region.shape}")
        print(f"  prob_idx 形状: {prob_idx.shape}")
        
        # 分割训练集和验证集
        curr_set = np.where(prob_idx != 38)[0]
        set_data = train_data[curr_set, :]
        set_region = train_region[curr_set, :]
        
        curr_val = np.where(prob_idx == 38)[0]
        val_data = train_data[curr_val, :]
        val_label = train_region[curr_val, :]
        
        print(f"分割后:")
        print(f"  set_data 形状: {set_data.shape}")
        print(f"  set_region 形状: {set_region.shape}")
        print(f"  val_data 形状: {val_data.shape}")
        print(f"  val_label 形状: {val_label.shape}")
        
        return {
            'set_data': set_data,
            'set_region': set_region,
            'val_data': val_data,
            'val_label': val_label
        }
    
    except Exception as e:
        print(f"加载数据集A时出错: {e}")
        return None

# 请确保更改下面的文件路径为您的实际路径
file_path = 'TRAIN38.mat'  # 请修改为真实路径
if os.path.exists(file_path):
    dataset_A = load_dataset_A(file_path)
else:
    print(f"文件不存在: {file_path}")

In [ ]:
# 加载数据集B（重组后的数据）
def load_dataset_B(base_dir):
    """加载重组后的数据集B"""
    print(f"加载数据集B: {base_dir}")
    
    # 检查目录是否存在
    if not os.path.exists(base_dir):
        print(f"目录不存在: {base_dir}")
        return None
    
    data_dirs = {
        'merged': os.path.join(base_dir, 'merged'),
        'train': os.path.join(base_dir, 'train'),
        'test': os.path.join(base_dir, 'test'),
        'val': os.path.join(base_dir, 'val')
    }
    
    # 检查所需的子目录是否都存在
    for name, dir_path in data_dirs.items():
        if not os.path.exists(dir_path):
            print(f"子目录不存在: {dir_path}")
            return None
    
    # 为每个子目录加载所有体素数据
    dataset_B = {}
    
    for name, dir_path in data_dirs.items():
        print(f"\n加载 {name} 数据...")
        
        # 获取所有标签文件
        voxel_files = glob.glob(os.path.join(dir_path, "label_*_count_*_voxels.npy"))
        
        if not voxel_files:
            print(f"  在 {name} 中未找到体素文件")
            continue
        
        # 加载所有体素数据并合并
        all_data = []
        all_labels = []
        
        for voxel_file in tqdm(voxel_files, desc=f"加载 {name} 数据"):
            # 从文件名提取标签ID
            filename = os.path.basename(voxel_file)
            parts = filename.split('_')
            label_id = int(parts[1])
            
            # 加载体素数据
            voxels = np.load(voxel_file)
            
            # 将标签ID扩展为与体素数据相同的行数
            labels = np.full((voxels.shape[0], 1), label_id)
            
            all_data.append(voxels)
            all_labels.append(labels)
        
        if all_data:
            combined_data = np.vstack(all_data)
            combined_labels = np.vstack(all_labels)
            
            dataset_B[name] = {
                'data': combined_data,
                'labels': combined_labels
            }
            
            print(f"  {name} 数据形状: {combined_data.shape}")
            print(f"  {name} 标签形状: {combined_labels.shape}")
    
    return dataset_B

# 请确保更改下面的目录路径为您的实际路径
restructured_base_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured'  # 请修改为真实路径
if os.path.exists(restructured_base_dir):
    dataset_B = load_dataset_B(restructured_base_dir)
else:
    print(f"目录不存在: {restructured_base_dir}")

In [ ]:
# 比较数据集A和数据集B的统计特性
def compare_datasets(dataset_A, dataset_B):
    """比较两个数据集的统计特性"""
    if dataset_A is None or dataset_B is None:
        print("无法比较数据集，请确保两个数据集已正确加载")
        return
    
    print("\n=== 数据集比较 ===")
    
    # 比较训练集数据
    if 'set_data' in dataset_A and 'train' in dataset_B:
        A_train = dataset_A['set_data']
        B_train = dataset_B['train']['data']
        
        print(f"训练集A形状: {A_train.shape}")
        print(f"训练集B形状: {B_train.shape}")
        
        # 检查特征数量是否相同
        if A_train.shape[1] == B_train.shape[1]:
            print("两个训练集的特征数量相同")
            
            # 计算每个特征的统计信息
            A_means = np.mean(A_train, axis=0)
            A_stds = np.std(A_train, axis=0)
            B_means = np.mean(B_train, axis=0)
            B_stds = np.std(B_train, axis=0)
            
            # 显示一些统计信息比较
            print("\n前10个特征的统计信息比较:")
            for i in range(min(10, A_train.shape[1])):
                print(f"特征 {i+1}:")
                print(f"  数据集A: 均值={A_means[i]:.4f}, 标准差={A_stds[i]:.4f}")
                print(f"  数据集B: 均值={B_means[i]:.4f}, 标准差={B_stds[i]:.4f}")
                print(f"  差异: 均值差={B_means[i]-A_means[i]:.4f}, 标准差比={B_stds[i]/A_stds[i]:.4f}")
            
            # 计算整体统计量的差异
            mean_diff = np.mean(np.abs(B_means - A_means))
            std_ratio = np.mean(B_stds / A_stds)
            
            print(f"\n整体统计差异:")
            print(f"  均值绝对差异平均值: {mean_diff:.4f}")
            print(f"  标准差比率平均值: {std_ratio:.4f}")
            
            # 可视化一些特征的分布
            n_features_to_plot = min(3, A_train.shape[1])
            fig, axes = plt.subplots(n_features_to_plot, 2, figsize=(14, 4*n_features_to_plot))
            
            for i in range(n_features_to_plot):
                # 直方图
                axes[i, 0].hist(A_train[:, i], bins=50, alpha=0.5, label='数据集A')
                axes[i, 0].hist(B_train[:, i], bins=50, alpha=0.5, label='数据集B')
                axes[i, 0].set_title(f'特征 {i+1} 分布')
                axes[i, 0].legend()
                
                # 箱线图
                box_data = [A_train[:, i], B_train[:, i]]
                axes[i, 1].boxplot(box_data, labels=['数据集A', '数据集B'])
                axes[i, 1].set_title(f'特征 {i+1} 箱线图')
            
            plt.tight_layout()
            plt.show()
        else:
            print(f"警告: 两个训练集的特征数量不同! A: {A_train.shape[1]}, B: {B_train.shape[1]}")
    else:
        print("无法比较训练集，缺少必要数据")

# 如果已加载数据集A和数据集B，则比较它们
if 'dataset_A' in globals() and 'dataset_B' in globals():
    if dataset_A is not None and dataset_B is not None:
        compare_datasets(dataset_A, dataset_B)

In [ ]:
# 优化的StandardScaler实现
class OptimizedScaler:
    """优化的StandardScaler，可以调整mean和std参数以最小化两个数据集之间的MSE"""
    
    def __init__(self, initial_mean=None, initial_std=None):
        """
        初始化优化的StandardScaler
        
        参数:
            initial_mean: 初始均值参数，可选
            initial_std: 初始标准差参数，可选
        """
        self.mean_ = initial_mean
        self.scale_ = initial_std
        self.n_features_ = None
    
    def fit(self, X, target_X, max_iter=1000, tol=1e-6, learning_rate=0.01, verbose=True):
        """
        通过梯度下降优化mean和std参数，使转换后的X与target_X之间的MSE最小
        
        参数:
            X: 原始数据集，形状为(n_samples, n_features)
            target_X: 目标数据集，形状与X相同
            max_iter: 最大迭代次数
            tol: 容忍度，如果MSE变化小于此值，则停止迭代
            learning_rate: 学习率
            verbose: 是否打印进度信息
        
        返回:
            self: 返回自身
        """
        # 检查输入
        X = np.asarray(X)
        target_X = np.asarray(target_X)
        
        if X.shape != target_X.shape:
            raise ValueError(f"X与target_X形状不同: {X.shape} vs {target_X.shape}")
        
        n_samples, n_features = X.shape
        self.n_features_ = n_features
        
        # 初始化参数
        if self.mean_ is None:
            self.mean_ = np.mean(X, axis=0)
        
        if self.scale_ is None:
            self.scale_ = np.std(X, axis=0)
            # 防止除以零
            self.scale_[self.scale_ == 0] = 1.0
        
        # 准备记录迭代过程
        self.loss_history_ = []
        
        # 梯度下降优化
        prev_loss = float('inf')
        
        for i in range(max_iter):
            # 计算当前转换
            X_scaled = (X - self.mean_) / self.scale_
            
            # 计算损失
            loss = np.mean((X_scaled - target_X) ** 2)
            self.loss_history_.append(loss)
            
            if verbose and (i % 10 == 0 or i == max_iter - 1):
                print(f"迭代 {i}, MSE: {loss:.6f}")
            
            # 检查收敛
            if abs(prev_loss - loss) < tol:
                if verbose:
                    print(f"收敛于迭代 {i}, MSE: {loss:.6f}")
                break
            
            prev_loss = loss
            
            # 计算梯度
            grad_mean = -2 * np.mean((X_scaled - target_X) * (1 / self.scale_), axis=0)
            grad_scale = -2 * np.mean((X_scaled - target_X) * (-1 * (X - self.mean_) / (self.scale_ ** 2)), axis=0)
            
            # 更新参数
            self.mean_ -= learning_rate * grad_mean
            self.scale_ -= learning_rate * grad_scale
            
            # 确保std始终为正数
            self.scale_ = np.maximum(self.scale_, 1e-10)
        
        return self
    
    def transform(self, X):
        """
        使用优化后的参数转换X
        
        参数:
            X: 要转换的数据集，形状为(n_samples, n_features)
        
        返回:
            X_scaled: 转换后的数据集
        """
        X = np.asarray(X)
        return (X - self.mean_) / self.scale_
    
    def fit_transform(self, X, target_X, **kwargs):
        """
        先fit再transform
        
        参数:
            X: 原始数据集
            target_X: 目标数据集
            **kwargs: 传递给fit方法的其他参数
        
        返回:
            X_scaled: 转换后的数据集
        """
        return self.fit(X, target_X, **kwargs).transform(X)
    
    def optimize_scipy(self, X, target_X, method='L-BFGS-B', verbose=True):
        """
        使用SciPy的优化器进行优化
        
        参数:
            X: 原始数据集，形状为(n_samples, n_features)
            target_X: 目标数据集，形状与X相同
            method: SciPy优化方法
            verbose: 是否打印进度信息
        
        返回:
            self: 返回自身
        """
        # 检查输入
        X = np.asarray(X)
        target_X = np.asarray(target_X)
        
        if X.shape != target_X.shape:
            raise ValueError(f"X与target_X形状不同: {X.shape} vs {target_X.shape}")
        
        n_samples, n_features = X.shape
        self.n_features_ = n_features
        
        # 初始化参数
        if self.mean_ is None:
            self.mean_ = np.mean(X, axis=0)
        
        if self.scale_ is None:
            self.scale_ = np.std(X, axis=0)
            # 防止除以零
            self.scale_[self.scale_ == 0] = 1.0
        
        # 准备记录迭代过程
        self.loss_history_ = []
        
        # 定义目标函数
        def objective(params):
            mean_params = params[:n_features]
            std_params = params[n_features:]
            
            # 确保std为正
            std_params = np.maximum(std_params, 1e-10)
            
            # 计算转换
            X_scaled = (X - mean_params) / std_params
            
            # 计算MSE
            loss = np.mean((X_scaled - target_X) ** 2)
            self.loss_history_.append(loss)
            
            return loss
        
        # 优化
        initial_params = np.concatenate([self.mean_, self.scale_])
        
        if verbose:
            print("开始SciPy优化...")
        
        result = minimize(
            objective, 
            initial_params, 
            method=method,
            options={'disp': verbose}
        )
        
        if verbose:
            print(f"优化完成: {result.message}")
            print(f"最终MSE: {result.fun:.6f}")
        
        # 更新参数
        self.mean_ = result.x[:n_features]
        self.scale_ = np.maximum(result.x[n_features:], 1e-10)
        
        return self

In [ ]:
# 应用优化的scaler
def apply_optimized_scaler(dataset_A, dataset_B):
    """应用优化的scaler将数据集A转换为接近数据集B"""
    if dataset_A is None or dataset_B is None:
        print("无法应用优化的scaler，请确保两个数据集已正确加载")
        return
    
    print("\n=== 应用优化的Scaler ===")
    
    # 获取训练数据
    if 'set_data' in dataset_A and 'train' in dataset_B:
        A_train = dataset_A['set_data']
        B_train = dataset_B['train']['data']
        
        # 检查特征数量是否相同
        if A_train.shape[1] == B_train.shape[1]:
            # 使用标准的StandardScaler作为基准
            print("\n使用标准StandardScaler:")
            standard_scaler = StandardScaler()
            standard_scaler.fit(A_train)
            A_train_std = standard_scaler.transform(A_train)
            
            # 计算MSE
            mse_standard = mean_squared_error(A_train_std, B_train)
            print(f"标准StandardScaler的MSE: {mse_standard:.6f}")
            
            # 使用优化的Scaler - 梯度下降方法
            print("\n使用优化的Scaler (梯度下降):")
            optimized_scaler_gd = OptimizedScaler()
            optimized_scaler_gd.fit(A_train, B_train, max_iter=300, learning_rate=0.01, verbose=True)
            A_train_opt_gd = optimized_scaler_gd.transform(A_train)
            
            # 计算MSE
            mse_optimized_gd = mean_squared_error(A_train_opt_gd, B_train)
            print(f"优化的Scaler (梯度下降) 的MSE: {mse_optimized_gd:.6f}")
            print(f"相对标准Scaler的改进: {(1 - mse_optimized_gd/mse_standard) * 100:.2f}%")
            
            # 使用优化的Scaler - SciPy优化方法
            print("\n使用优化的Scaler (SciPy优化):")
            optimized_scaler_scipy = OptimizedScaler()
            optimized_scaler_scipy.optimize_scipy(A_train, B_train, verbose=True)
            A_train_opt_scipy = optimized_scaler_scipy.transform(A_train)
            
            # 计算MSE
            mse_optimized_scipy = mean_squared_error(A_train_opt_scipy, B_train)
            print(f"优化的Scaler (SciPy优化) 的MSE: {mse_optimized_scipy:.6f}")
            print(f"相对标准Scaler的改进: {(1 - mse_optimized_scipy/mse_standard) * 100:.2f}%")
            
            # 可视化比较结果
            print("\n可视化比较:")
            
            # 绘制损失曲线
            plt.figure(figsize=(10, 6))
            plt.plot(optimized_scaler_gd.loss_history_, label='梯度下降损失')
            plt.plot(optimized_scaler_scipy.loss_history_, label='SciPy优化损失')
            plt.axhline(y=mse_standard, color='r', linestyle='-', label='标准Scaler MSE')
            plt.xlabel('迭代次数')
            plt.ylabel('MSE')
            plt.title('优化过程中的MSE变化')
            plt.legend()
            plt.grid(True)
            plt.show()
            
            # 比较分布
            n_features_to_plot = min(3, A_train.shape[1])
            fig, axes = plt.subplots(n_features_to_plot, 1, figsize=(12, 4*n_features_to_plot))
            
            for i in range(n_features_to_plot):
                feature_idx = i
                
                if n_features_to_plot == 1:
                    ax = axes
                else:
                    ax = axes[i]
                
                ax.hist(A_train[:, feature_idx], bins=50, alpha=0.3, label='原始数据集A')
                ax.hist(A_train_std[:, feature_idx], bins=50, alpha=0.3, label='标准Scaler')
                ax.hist(A_train_opt_gd[:, feature_idx], bins=50, alpha=0.3, label='优化Scaler (GD)')
                ax.hist(A_train_opt_scipy[:, feature_idx], bins=50, alpha=0.3, label='优化Scaler (SciPy)')
                ax.hist(B_train[:, feature_idx], bins=50, alpha=0.3, label='目标数据集B')
                ax.set_title(f'特征 {feature_idx+1} 分布比较')
                ax.legend()
            
            plt.tight_layout()
            plt.show()
            
            # 返回最佳的scaler
            if mse_optimized_gd < mse_optimized_scipy:
                print("\n梯度下降方法表现更好")
                return optimized_scaler_gd
            else:
                print("\nSciPy优化方法表现更好")
                return optimized_scaler_scipy
        else:
            print(f"警告: 两个训练集的特征数量不同! A: {A_train.shape[1]}, B: {B_train.shape[1]}")
    else:
        print("无法应用优化的scaler，缺少必要数据")

# 如果已加载数据集A和数据集B，则应用优化的scaler
if 'dataset_A' in globals() and 'dataset_B' in globals():
    if dataset_A is not None and dataset_B is not None:
        best_scaler = apply_optimized_scaler(dataset_A, dataset_B)

In [ ]:
# 验证优化的scaler在验证集上的表现
def validate_optimized_scaler(dataset_A, dataset_B, best_scaler):
    """验证优化的scaler在验证集上的表现"""
    if dataset_A is None or dataset_B is None or best_scaler is None:
        print("无法验证优化的scaler，请确保数据集和scaler已正确加载")
        return
    
    print("\n=== 验证优化的Scaler ===")
    
    # 获取验证数据
    if 'val_data' in dataset_A and 'val' in dataset_B:
        A_val = dataset_A['val_data']
        B_val = dataset_B['val']['data']
        
        # 检查特征数量是否相同
        if A_val.shape[1] == B_val.shape[1]:
            # 使用标准的StandardScaler
            standard_scaler = StandardScaler()
            standard_scaler.fit(A_val)
            A_val_std = standard_scaler.transform(A_val)
            
            # 计算MSE
            mse_standard = mean_squared_error(A_val_std, B_val)
            print(f"标准StandardScaler在验证集上的MSE: {mse_standard:.6f}")
            
            # 使用优化的Scaler
            A_val_opt = best_scaler.transform(A_val)
            
            # 计算MSE
            mse_optimized = mean_squared_error(A_val_opt, B_val)
            print(f"优化的Scaler在验证集上的MSE: {mse_optimized:.6f}")
            print(f"相对标准Scaler的改进: {(1 - mse_optimized/mse_standard) * 100:.2f}%")
            
            # 可视化比较结果
            n_features_to_plot = min(3, A_val.shape[1])
            fig, axes = plt.subplots(n_features_to_plot, 1, figsize=(12, 4*n_features_to_plot))
            
            for i in range(n_features_to_plot):
                feature_idx = i
                
                if n_features_to_plot == 1:
                    ax = axes
                else:
                    ax = axes[i]
                
                ax.hist(A_val[:, feature_idx], bins=50, alpha=0.3, label='原始验证集A')
                ax.hist(A_val_std[:, feature_idx], bins=50, alpha=0.3, label='标准Scaler')
                ax.hist(A_val_opt[:, feature_idx], bins=50, alpha=0.3, label='优化Scaler')
                ax.hist(B_val[:, feature_idx], bins=50, alpha=0.3, label='目标验证集B')
                ax.set_title(f'特征 {feature_idx+1} 验证集分布比较')
                ax.legend()
            
            plt.tight_layout()
            plt.show()
            
            return {
                'mse_standard': mse_standard,
                'mse_optimized': mse_optimized,
                'improvement': (1 - mse_optimized/mse_standard) * 100
            }
        else:
            print(f"警告: 两个验证集的特征数量不同! A: {A_val.shape[1]}, B: {B_val.shape[1]}")
    else:
        print("无法验证优化的scaler，缺少必要数据")

# 如果已加载数据集A、数据集B和最佳scaler，则验证优化的scaler
if 'dataset_A' in globals() and 'dataset_B' in globals() and 'best_scaler' in globals():
    if dataset_A is not None and dataset_B is not None and best_scaler is not None:
        validation_results = validate_optimized_scaler(dataset_A, dataset_B, best_scaler)

In [ ]:
# 保存和加载优化后的scaler
def save_optimized_scaler(scaler, file_path):
    """保存优化后的scaler到文件"""
    if scaler is None:
        print("无法保存scaler，请确保scaler已正确创建")
        return False
    
    # 准备要保存的数据
    data = {
        'mean_': scaler.mean_,
        'scale_': scaler.scale_,
        'n_features_': scaler.n_features_,
        'loss_history_': scaler.loss_history_
    }
    
    try:
        np.savez(file_path, **data)
        print(f"优化后的scaler已保存到: {file_path}")
        return True
    except Exception as e:
        print(f"保存scaler时出错: {e}")
        return False

def load_optimized_scaler(file_path):
    """从文件加载优化后的scaler"""
    try:
        data = np.load(file_path)
        scaler = OptimizedScaler()
        scaler.mean_ = data['mean_']
        scaler.scale_ = data['scale_']
        scaler.n_features_ = data['n_features_']
        scaler.loss_history_ = data['loss_history_']
        
        print(f"优化后的scaler已从 {file_path} 加载")
        return scaler
    except Exception as e:
        print(f"加载scaler时出错: {e}")
        return None

# 如果已创建最佳scaler，则保存它
if 'best_scaler' in globals() and best_scaler is not None:
    save_optimized_scaler(best_scaler, 'optimized_scaler.npz')
    
    # 测试加载
    loaded_scaler = load_optimized_scaler('optimized_scaler.npz')
    
    if loaded_scaler is not None:
        print("\n验证加载的scaler:")
        print(f"均值前10个值: {loaded_scaler.mean_[:10]}")
        print(f"标准差前10个值: {loaded_scaler.scale_[:10]}")

In [ ]:
# 使用优化的scaler示例
def scaler_usage_example():
    """展示如何使用优化的scaler"""
    print("\n=== 优化的Scaler使用示例 ===")
    
    # 创建模拟数据
    np.random.seed(42)
    X = np.random.randn(100, 5)  # 原始数据
    target_mean = np.array([1, 2, 3, 4, 5])
    target_std = np.array([0.5, 1, 1.5, 2, 2.5])
    target_X = np.random.randn(100, 5) * target_std + target_mean  # 目标数据
    
    # 使用标准的StandardScaler
    standard_scaler = StandardScaler()
    standard_scaler.fit(X)
    X_std = standard_scaler.transform(X)
    
    # 计算MSE
    mse_standard = mean_squared_error(X_std, target_X)
    print(f"标准StandardScaler的MSE: {mse_standard:.6f}")
    
    # 使用优化的Scaler
    optimized_scaler = OptimizedScaler()
    optimized_scaler.optimize_scipy(X, target_X, verbose=True)
    X_opt = optimized_scaler.transform(X)
    
    # 计算MSE
    mse_optimized = mean_squared_error(X_opt, target_X)
    print(f"优化的Scaler的MSE: {mse_optimized:.6f}")
    print(f"相对标准Scaler的改进: {(1 - mse_optimized/mse_standard) * 100:.2f}%")
    
    # 比较参数
    print("\n参数比较:")
    print("标准Scaler均值:")
    print(standard_scaler.mean_)
    print("优化Scaler均值:")
    print(optimized_scaler.mean_)
    print("目标均值:")
    print(target_mean)
    
    print("\n标准Scaler标准差:")
    print(standard_scaler.scale_)
    print("优化Scaler标准差:")
    print(optimized_scaler.scale_)
    print("目标标准差:")
    print(target_std)
    
    # 可视化比较
    plt.figure(figsize=(10, 6))
    plt.plot(optimized_scaler.loss_history_, label='优化过程中的MSE')
    plt.axhline(y=mse_standard, color='r', linestyle='-', label='标准Scaler MSE')
    plt.xlabel('迭代次数')
    plt.ylabel('MSE')
    plt.title('优化过程中的MSE变化')
    plt.legend()
    plt.grid(True)
    plt.show()

# 运行使用示例
scaler_usage_example()

print("\n=== 总结 ===")
print("""
本实验实现了一个优化的StandardScaler，可以调整其均值和标准差参数，使得转换后的数据集与目标数据集之间的MSE最小。实验中使用了两种优化方法：
1. 梯度下降法：通过计算MSE对均值和标准差的梯度，迭代更新参数
2. SciPy优化：使用SciPy库的optimize.minimize函数进行优化

实验结果表明，优化后的scaler可以显著降低转换后数据集与目标数据集之间的MSE，相对于标准的StandardScaler有明显改进。

优化后的scaler可以保存到文件并在需要时加载，方便在不同的数据处理流程中复用。

这种方法特别适用于需要将一个数据集的分布调整为接近另一个数据集的场景，例如跨域适应、迁移学习等。
""")